In [4]:
!mkdir -p n5k/metadata
!gsutil -m cp "gs://nutrition5k_dataset/nutrition5k_dataset/metadata/*" n5k/metadata/
!gsutil ls gs://nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/ > n5k/overhead_dirs.txt
!ls n5k/metadata

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying gs://nutrition5k_dataset/nutrition5k_dataset/metadata/dish_metadata_cafe1.csv...
Copying gs://nutrition5k_dataset/nutrition5k_dataset/metadata/dish_metadata_cafe2.csv...
Copying gs://nutrition5k_dataset/nutrition5k_dataset/metadata/ingredients_metadata.csv...
/ [3/3 files][  2.2 MiB/  2.2 MiB] 100% Done                                    
Operation completed over 3 objects/2.2 MiB.                                      
dish_metadata_cafe1.csv  dish_metadata_cafe2.csv  ingredients_metadata.csv


In [7]:
"""
Nutrition5k 数据集审计：能否按食材划分"见过 / 没见过"？
在 Colab 里运行。先下载 metadata（很小，几 MB）：

!mkdir -p n5k/metadata
!gsutil -m cp "gs://nutrition5k_dataset/nutrition5k_dataset/metadata/*" n5k/metadata/
!gsutil ls gs://nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/ > n5k/overhead_dirs.txt
"""
import csv, re
from collections import defaultdict
import pandas as pd

ROOT = "n5k"
META_FILES = [f"{ROOT}/metadata/dish_metadata_cafe1.csv",
              f"{ROOT}/metadata/dish_metadata_cafe2.csv"]
THRESHOLDS = [0.0, 0.3, 0.5]        # 0.0 = 只要含有就算
GROUP_GAP_SEC = 600                  # 增量扫描分组的时间窗口，需要人工核对后调整

# ---------- 1. 解析不定长 CSV ----------
def parse_metadata(path, cafe):
    dishes, ingrs = [], []
    with open(path) as f:
        for row in csv.reader(f):
            if not row or not row[0].startswith("dish_"):
                continue
            d = dict(dish_id=row[0], cafe=cafe,
                     cal=float(row[1]), mass=float(row[2]), fat=float(row[3]),
                     carb=float(row[4]), protein=float(row[5]))
            rest = row[6:]
            # 每种食材 7 个字段：id, name, grams, cal, fat, carb, protein
            for i in range(0, len(rest) - len(rest) % 7, 7):
                ingrs.append(dict(dish_id=row[0], ingr_id=rest[i],
                                  name=rest[i + 1].strip().lower(),
                                  grams=float(rest[i + 2]), ingr_cal=float(rest[i + 3])))
            d["n_ingr"] = len(rest) // 7
            dishes.append(d)
    return dishes, ingrs

D, I = [], []
for k, p in enumerate(META_FILES, 1):
    d, i = parse_metadata(p, f"cafe{k}")
    D += d; I += i
dish = pd.DataFrame(D).drop_duplicates("dish_id").set_index("dish_id")
ingr = pd.DataFrame(I)

# ---------- 2. 只保留有俯拍 RGB-D 的菜 ----------
with open(f"{ROOT}/overhead_dirs.txt") as f:
    overhead = {m.group(0) for m in re.finditer(r"dish_\d+", f.read())}
dish["has_depth"] = dish.index.isin(overhead)
print(f"总菜数 {len(dish)}，有俯拍深度 {dish.has_depth.sum()}")
dish = dish[dish.has_depth & (dish.mass > 0)]
ingr = ingr[ingr.dish_id.isin(dish.index)]

# ---------- 3. 基本统计 ----------
ingr["frac"] = ingr.grams / ingr.dish_id.map(dish.mass)
ingr["cal_frac"] = ingr.ingr_cal / ingr.dish_id.map(dish.cal).replace(0, float("nan"))
summary = (ingr.groupby("name")
           .agg(n_dishes=("dish_id", "nunique"), total_g=("grams", "sum"),
                n_ge30=("frac", lambda x: (x >= 0.3).sum()))
           .sort_values("total_g", ascending=False))
summary.to_csv("ingredient_summary.csv")
print(f"\n共 {len(summary)} 种食材。按总质量前 40：")
print(summary.head(40).to_string())

# ---------- 4. 食材大类（看完 ingredient_summary.csv 后手动修改！） ----------
CATEGORIES = {
    "seafood": r"^(?:cod|fish|salmon|shrimp|tuna|tuna salad)$",
    "chicken": r"^(?:chicken|chicken breast|chicken thighs|grilled chicken|chicken salad)$",
    "egg": r"^(?:egg whites|eggs|poached eggs|scrambled eggs)$",
    "fruit": r"^(?:apple|berries|blueberries|cantaloupe|grapes|honeydew melons|pears|pineapple|strawberries|watermelon)$",
}
for cat, pat in CATEGORIES.items():
    hit = ingr[ingr.name.str.contains(pat, regex=True)]
    matched = sorted(hit.name.unique())
    frac = hit.groupby("dish_id").frac.sum()       # 同一大类多种食材，占比相加
    counts = {t: int((frac > t).sum() if t == 0 else (frac >= t).sum()) for t in THRESHOLDS}
    print(f"\n[{cat}] 匹配到 {len(matched)} 种食材: {matched[:12]}{' ...' if len(matched) > 12 else ''}")
    print("   菜数 " + "  ".join(f"≥{int(t*100)}%: {c}" if t else f"含有: {c}" for t, c in counts.items()))
    dish[f"frac_{cat}"] = frac.reindex(dish.index).fillna(0)

# ---------- 5. 重建增量扫描分组（防止同一盘菜跨划分泄漏） ----------
# 启发式：同一 cafe 内按时间戳排序，若下一道菜时间间隔 < GROUP_GAP_SEC、
# 食材集合包含上一道、总质量更大，则视为同一盘菜的下一次扫描。
sets = ingr.groupby("dish_id").ingr_id.apply(set)
dish["ts"] = dish.index.str.extract(r"(\d+)", expand=False).astype(int).values
group_id, gid = {}, 0
for cafe, sub in dish.sort_values("ts").groupby("cafe"):
    prev = None
    for did, r in sub.iterrows():
        if prev is not None:
            p_id, p_r = prev
            same = (r.ts - p_r.ts < GROUP_GAP_SEC and r.mass > p_r.mass
                    and sets.get(p_id, set()) <= sets.get(did, set()))
            if not same:
                gid += 1
        group_id[did] = gid
        prev = (did, r)
dish["plate_group"] = pd.Series(group_id)
sizes = dish.plate_group.value_counts()
print(f"\n重建出 {len(sizes)} 个盘子分组；每组扫描次数分布：")
print(sizes.value_counts().sort_index().to_string())

# 关键检查：被选为"没见过"的菜，同组里有多少次扫描不满足条件（会被分到训练集 → 泄漏）
for cat in CATEGORIES:
    sel = dish[f"frac_{cat}"] >= 0.3
    groups = dish.loc[sel, "plate_group"].unique()
    in_groups = dish.plate_group.isin(groups)
    leak = (in_groups & ~sel).sum()
    print(f"[{cat}] ≥30% 的菜 {sel.sum()} 道，分布在 {len(groups)} 个盘子；"
          f"同盘但 <30% 的扫描 {leak} 道（这些必须一起丢掉或一起划分）")

dish.to_csv("dish_manifest_draft.csv")
print("\n已保存 ingredient_summary.csv 和 dish_manifest_draft.csv")

总菜数 5006，有俯拍深度 3490

共 241 种食材。按总质量前 40：
                  n_dishes       total_g  n_ge30
name                                            
egg whites             179  43264.333297     144
chicken                312  37588.691288     188
scrambled eggs         229  29816.357124     195
cauliflower            251  22671.210358     134
broccoli               418  21837.502638     102
cheese pizza           262  17900.000000     129
fish                   188  17388.465612     102
berries                181  17350.000000     112
caesar salad           185  16713.000000     109
sweet potato           197  15909.329730      68
carrot                 447  15545.645275      80
pork                   181  15154.189049      97
cherry tomatoes        483  14682.710501      78
salmon                 132  14663.004088      77
beef                   173  13200.023249      67
apple                  174  12904.695806      86
pineapple              184  12615.896801      64
roasted potatoes       171  

In [8]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/nutriscan
!cp -r n5k n5k_split_audit.py ingredient_summary.csv dish_manifest_draft.csv /content/drive/MyDrive/nutriscan/

Mounted at /content/drive
cp: cannot stat 'n5k_split_audit.py': No such file or directory
